In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error, r2_score

# Загружаем уже очищенные нами данные
df = pd.read_csv('../data/processed/cleaned_data.csv')

features = ['Age', 'Sales', 'Quantity', 'Discount', 'Shipping Cost', 'Browsing Time (min)', 'Total_Engagement']
X = df[features]
y = df['Profit']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [2]:
# Уменьшаем количество признаков до 3-х
pca = PCA(n_components=3)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

print(f"Размерность данных после PCA: {X_train_pca.shape}")

Размерность данных после PCA: (41027, 3)


In [3]:
results = []

models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(),
    'Lasso': Lasso(),
    'Decision Tree': DecisionTreeRegressor(random_state=42)
}

# Обучаем простые модели
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results.append({
        'Model': name,
        'MAE': mean_absolute_error(y_test, preds),
        'R2': r2_score(y_test, preds)
    })

# ПЕРЕБОР ГИПЕРПАРАМЕТРОВ для Random Forest
rf_params = {'n_estimators': [50, 100], 'max_depth': [5, 10, None]}
grid_rf = GridSearchCV(RandomForestRegressor(random_state=42), rf_params, cv=3, scoring='r2')
grid_rf.fit(X_train, y_train)

best_rf = grid_rf.best_estimator_
rf_preds = best_rf.predict(X_test)
results.append({
    'Model': 'Random Forest (Tuned)',
    'MAE': mean_absolute_error(y_test, rf_preds),
    'R2': r2_score(y_test, rf_preds)
})

In [4]:
results_df = pd.DataFrame(results)
print("Финальная таблица экспериментов:")
display(results_df)

Финальная таблица экспериментов:


,Model,MAE,R2
0,Linear Regression,2.557210e-01,0.999964
1,Ridge,2.557907e-01,0.999964
2,Lasso,4.616528e-01,0.999860
3,Decision Tree,1.133786e-13,1.000000
4,Random Forest (Tuned),1.639856e-04,1.000000


Вывод по моделированию:

В ходе экспериментов было протестировано 5 различных моделей.

Внедрен перебор гиперпараметров для Random Forest через GridSearchCV.

Использован метод PCA для анализа возможности уменьшения размерности.

Финальный выбор: Decision Tree и Random Forest показали идентичные результаты (R2=1.0), что обусловлено структурой данных. Для продакшена выбираем Линейную регрессию, так как она проще и быстрее в вычислениях.